# Representaciones vectoriales de textos (*Embbedings*)

- Usar embbedings para representar textos
- Usar aritmética de embeddings
- Entrenar modelos usando embeddings

*Word2Vec* es una técnica de representación de texto que transforma palabras en vectores numéricos de forma que aquellas con significados similares queden cerca en un espacio multidimensional. 🧠✨ La intuición detrás del modelo es que las palabras que aparecen en contextos parecidos tienden a tener significados relacionados —por ejemplo, *“rey”* y *“reina”* compartirán coordenadas cercanas. En Python, la librería *gensim* facilita su uso, permitiendo cargar, entrenar y explorar modelos con comandos simples como `KeyedVectors.load_word2vec_format()`. En este caso, se utiliza un modelo **preentrenado en español**, disponible como `w2vec/SBW-vectors-300-min5.txt`, que contiene vectores de 300 dimensiones generados a partir de un enorme corpus del idioma. Gracias a estos embeddings preentrenados, es posible analizar similitudes semánticas, relaciones entre palabras y realizar operaciones vectoriales sin necesidad de entrenar un modelo desde cero. 🇪🇸📊


## Setup

In [ ]:
!pip install gensim

In [ ]:
!pip install  transformers torch  xgboost tqdm

In [1]:
!curl -L -o pretrained-word-vectors-for-spanish.zip https://www.kaggle.com/api/v1/datasets/download/rtatman/pretrained-word-vectors-for-spanish


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1004M  100 1004M    0     0  20.9M      0  0:00:48  0:00:48 --:--:-- 22.0M


In [2]:
%%shell
mkdir w2vec
unzip pretrained-word-vectors-for-spanish.zip -d w2vec/

Archive:  pretrained-word-vectors-for-spanish.zip
  inflating: w2vec/SBW-vectors-300-min5.txt  


In [36]:
from gensim.models import KeyedVectors
import numpy as np
from scipy.spatial.distance import cosine
import pandas as pd



In [44]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [45]:
# 2️⃣ Cargar modelo BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3️⃣ Función para obtener embeddings de una descripción (promedio de tokens)
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # Promedio de los embeddings de la última capa oculta
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

## Cargado de embeddings preentrenados

In [3]:

# Ruta al archivo descargado
file_path = "w2vec/SBW-vectors-300-min5.txt"  # ajusta si tu carpeta tiene otro nombre

# Cargar modelo (formato texto)
print("Cargando modelo, esto puede tardar un poco...")
model = KeyedVectors.load_word2vec_format(file_path, binary=False)

print("✅ Modelo cargado con éxito")
print("Dimensión de los embeddings:", model.vector_size)
print("Palabras en el vocabulario:", len(model.key_to_index))

Cargando modelo, esto puede tardar un poco...
✅ Modelo cargado con éxito
Dimensión de los embeddings: 300
Palabras en el vocabulario: 1000653


## Relaciones semánticas y aritmética de embeddings

Las *relaciones semánticas* en los embeddings surgen porque cada palabra se representa como un vector en un espacio donde las distancias reflejan su significado. 🧭 Palabras que aparecen en contextos similares tendrán vectores cercanos, mientras que las que se usan en contextos distintos estarán más alejadas.  

Una forma común de medir esta relación es mediante la **similitud del coseno**, que compara el ángulo entre dos vectores en lugar de su magnitud. Si dos vectores apuntan en direcciones muy parecidas, el coseno será cercano a 1, lo que indica una fuerte relación semántica. Su fórmula es:  

$$
\text{Similitud\_coseno}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}
$$  

Esto permite evaluar qué tan similares son dos palabras como *“feliz”* y *“alegre”*, o qué tan diferentes son *“feliz”* y *“triste”*. 🌈  

La *aritmética de embeddings* aprovecha esta propiedad: al sumar y restar vectores, se pueden descubrir relaciones semánticas latentes, como en el clásico ejemplo  
$$\text{rey} - \text{hombre} + \text{mujer} \approx \text{reina}$$  
lo que demuestra que el modelo no solo aprende palabras, sino también **conceptos y relaciones entre ellas**. 🤖💬


<center>
  <img src="https://media.telefonicatech.com/telefonicatech/uploads/2021/1/40390_embeddings_general.png" alt="Regresion Descenso de Gradiente" style="max-width:50%; height:auto;"  width="100%">
</center>

### **Ejemplos**

In [13]:
model.most_similar("feliz", topn=10)

[('contenta', 0.7562954425811768),
 ('felices', 0.7461491823196411),
 ('contento', 0.7461214661598206),
 ('ilusionada', 0.7037598490715027),
 ('emocionada', 0.6960131525993347),
 ('apapachada', 0.6946418285369873),
 ('emocionado', 0.6868844628334045),
 ('orgulloso', 0.6765345931053162),
 ('felicidad', 0.6745385527610779),
 ('supercontenta', 0.6721425652503967)]

In [14]:
# 1️⃣ Relaciones de género
print("👑 rey - hombre + mujer ≈")
print(model.most_similar(positive=["rey", "mujer"], negative=["hombre"], topn=5))

👑 rey - hombre + mujer ≈
[('reina', 0.7493032217025757), ('consorte', 0.703425943851471), ('princesa', 0.6861547231674194), ('Olofsdotter', 0.6481983065605164), ('reyes', 0.6391469836235046)]


In [15]:
# 2️⃣ Países y capitales
print("🌍 España - Madrid + Francia ≈")
print(model.most_similar(positive=["Francia", "Madrid"], negative=["España"], topn=5))


🌍 España - Madrid + Francia ≈
[('París', 0.7476182579994202), ('Lyon', 0.7073330283164978), ('Marsella', 0.699757993221283), ('Burdeos', 0.6509923338890076), ('Lille', 0.650432288646698)]


In [18]:
# 3️⃣ Singular y plural
print("🍎 manzana - una + muchas ≈")
print(model.most_similar(positive=["manzana", "muchas"], negative=["una"], topn=5))

🍎 manzana - una + muchas ≈
[('muchísimas', 0.5300258994102478), ('tantas', 0.5284314155578613), ('algunas', 0.508005678653717), ('demasiadas', 0.4618566930294037), ('Muchas', 0.45585915446281433)]


In [19]:
# 4️⃣ Presente y pasado
print("⌛ correr - corre + corrió ≈")
print(model.most_similar(positive=["corrió", "corre"], negative=["correr"], topn=5))

⌛ correr - corre + corrió ≈
[('correrá', 0.5755766034126282), ('corría', 0.5619832277297974), ('corrieron', 0.519588828086853), ('corren', 0.4657655954360962), ('estuvo', 0.4546167850494385)]


In [20]:
# 5️⃣ Masculino y femenino
print("🧑 amigo - hombre + mujer ≈")
print(model.most_similar(positive=["amigo", "mujer"], negative=["hombre"], topn=5))

🧑 amigo - hombre + mujer ≈
[('amiga', 0.8069695830345154), ('hermana', 0.7288695573806763), ('sobrina', 0.7073289155960083), ('esposa', 0.7045512795448303), ('tía', 0.684664249420166)]


In [21]:
# 6️⃣ País y gentilicio
print("🇲🇽 México - mexicano + España ≈")
print(model.most_similar(positive=["España", "mexicano"], negative=["México"], topn=5))

🇲🇽 México - mexicano + España ≈
[('español', 0.742891788482666), ('española', 0.6267584562301636), ('madrileño', 0.549006998538971), ('espańol', 0.5396885871887207), ('cántabro', 0.5340784192085266)]


In [22]:
# 7️⃣ Estaciones del año
print("🌸 primavera - frío + calor ≈")
print(model.most_similar(positive=["primavera", "frio"], negative=["calor"], topn=5))


🌸 primavera - frío + calor ≈
[('otoño', 0.6644449234008789), ('invierno', 0.561501145362854), ('verano', 0.559830367565155), ('Florecen', 0.5194669365882874), ('boreal', 0.4829379618167877)]


In [23]:

# 8️⃣ Contrastes de emociones
print("😊 feliz - alegría + tristeza ≈")
print(model.most_similar(positive=["feliz", "tristeza"], negative=["alegría"], topn=5))

😊 feliz - alegría + tristeza ≈
[('infeliz', 0.676829993724823), ('triste', 0.6600649952888489), ('apenado', 0.6503477692604065), ('apenada', 0.6317839622497559), ('apesadumbrado', 0.5985892415046692)]


In [24]:
# 9️⃣ Relación de objetos y lugares
print("📚 escuela - estudiante + enfermo ≈")
print(model.most_similar(positive=["escuela", "medicos"], negative=["profesor"], topn=5))

📚 escuela - estudiante + enfermo ≈
[('farmaceutas', 0.4900882840156555), ('huerfanos', 0.47184130549430847), ('ortopedistas', 0.4662727117538452), ('ninos', 0.4640234410762787), ('clinica', 0.463959276676178)]


In [25]:
# 🔟 Conceptos temporales
print("🕐 día - sol + noche ≈")
print(model.most_similar(positive=["noche", "sol"], negative=["día"], topn=5))


🕐 día - sol + noche ≈
[('luna', 0.5530824065208435), ('abrasador', 0.5268511772155762), ('calcinante', 0.5250343680381775), ('brisa', 0.5170868039131165), ('oscuridad', 0.5161758661270142)]


## Representación media de un texto

La representación promedio de los *embeddings* de un texto se basa en la idea de que cada palabra aporta una pequeña parte del significado general. 🧠✨ Al calcular el promedio de los vectores de todas las palabras, obtenemos un punto en el espacio semántico que resume el “centro de significado” del texto completo. Este enfoque funciona porque los *embeddings* ya codifican relaciones semánticas: si muchas palabras en una reseña están asociadas con emociones positivas, su promedio estará más cerca de otras regiones del espacio donde se encuentran conceptos similares. Aunque se pierda el orden y parte del contexto, esta técnica ofrece una forma simple y efectiva de representar oraciones o documentos con un solo vector que refleja su tono o tema general. 📊💬


In [27]:
def vector_promedio(texto, modelo):
    palabras = texto.lower().split()
    vectores = [modelo[w] for w in palabras if w in modelo.key_to_index]
    if len(vectores) == 0:
        return np.zeros(modelo.vector_size)
    return np.mean(vectores, axis=0)

In [28]:
def analizar_sentimiento_texto(texto,modelo,sim_pos,sim_neg):
    v_texto = vector_promedio(texto, modelo)
    sim_pos = 1 - cosine(v_texto, v_pos)
    sim_neg = 1 - cosine(v_texto, v_neg)
    print(f"Texto: {texto}")
    print(f"Similitud con centro positivo: {sim_pos:.3f}")
    print(f"Similitud con centro negativo: {sim_neg:.3f}")
    if sim_pos > sim_neg:
        print("➡️ Clasificada como POSITIVA 😄\n")
    else:
        print("➡️ Clasificada como NEGATIVA 😞\n")

In [29]:
positivas = ["excelente", "bueno", "agradable", "suave", "perfecto", "recomendado"]
negativas = ["malo", "irritación", "reseca", "defectuoso", "decepcionante", "caro"]

v_pos = np.mean([model[w] for w in positivas if w in model.key_to_index], axis=0)
v_neg = np.mean([model[w] for w in negativas if w in model.key_to_index], axis=0)

In [33]:
analizar_sentimiento_texto("La crema me dejó la piel muy suave y luminosa",model,positivas,negativas)
analizar_sentimiento_texto("El aroma no es bueno, decepción",model,positivas,negativas)
analizar_sentimiento_texto("Cumple, pero no noté grandes resultados",model,positivas,negativas)

Texto: La crema me dejó la piel muy suave y luminosa
Similitud con centro positivo: 0.670
Similitud con centro negativo: 0.612
➡️ Clasificada como POSITIVA 😄

Texto: El aroma no es bueno, decepción
Similitud con centro positivo: 0.599
Similitud con centro negativo: 0.640
➡️ Clasificada como NEGATIVA 😞

Texto: Cumple, pero no noté grandes resultados
Similitud con centro positivo: 0.579
Similitud con centro negativo: 0.637
➡️ Clasificada como NEGATIVA 😞



## Uso de textos en modelos de regresión

🍷 **Wine Reviews (Wine Enthusiast)**  

El dataset proviene de la revista *Wine Enthusiast*, una publicación especializada en reseñas y calificaciones de vinos de distintas regiones del mundo. Cada registro corresponde a un vino con su descripción sensorial redactada por expertos catadores, junto con información adicional como país de origen, variedad de uva, precio y puntuación asignada.  

Este conjunto de datos es ampliamente utilizado para análisis de texto, minería de opiniones y modelos predictivos en tareas de regresión o clasificación.  

---

🎯 **Descripción de la tarea**  

El objetivo es **predecir la puntuación (score)** que un vino recibe, basándose en la información disponible —principalmente la descripción textual del producto y variables complementarias como el país, el precio o la variedad de uva.  

En otras palabras:  
> “¿Podemos estimar la calidad de un vino a partir de cómo lo describen los expertos?” 🍇  

La tarea se plantea como un **problema de regresión**, donde la variable dependiente es la puntuación del vino, expresada en una escala de 0 a 100.  

---

📘 **Diccionario de datos (principales variables)**  

| Columna        | Tipo de dato | Descripción |
|----------------|--------------|-------------|
| `description`  | texto        | Reseña del vino escrita por un catador. Es la principal fuente de información para el análisis de lenguaje. |
| `points`       | numérico     | Puntuación del vino (de 80 a 100 puntos). Es la variable objetivo a predecir. |
| `price`        | numérico     | Precio del vino en dólares. Puede correlacionar con la calidad percibida. |
| `country`      | categórico   | País de origen del vino (ej. *Italy*, *France*, *US*). |
| `province`     | categórico   | Región o provincia dentro del país productor. |
| `variety`      | categórico   | Tipo de uva utilizada (ej. *Pinot Noir*, *Chardonnay*). |
| `winery`       | categórico   | Nombre del productor o bodega. |
| `designation`  | texto/categórico | Nombre o etiqueta especial del vino (opcional). |
| `region_1`, `region_2` | categórico | Subregiones dentro del país o provincia (algunas pueden estar vacías). |
| `taster_name`  | categórico   | Nombre del catador que escribió la reseña. |
| `title`        | texto        |


### Carga de datos y exploración

In [34]:
!curl -L -o wine-reviews.zip https://www.kaggle.com/api/v1/datasets/download/zynicide/wine-reviews

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 50.8M  100 50.8M    0     0   9.9M      0  0:00:05  0:00:05 --:--:-- 14.6M


In [35]:
%%shell
mkdir wines
unzip wine-reviews.zip -d wines/

Archive:  wine-reviews.zip
  inflating: wines/winemag-data-130k-v2.csv  
  inflating: wines/winemag-data-130k-v2.json  
  inflating: wines/winemag-data_first150k.csv  


In [37]:
wines_df= pd.read_csv('wines/winemag-data_first150k.csv')

In [38]:
wines_df.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery
0,0,US,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz
1,1,Spain,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,NaN,Tinta de Toro,Bodega Carmen Rodríguez
2,2,US,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley
3,3,US,"This spent 20 months in 30% new French oak, an...",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi
4,4,France,"This is the top wine from La Bégude, named aft...",La Brûlade,95,66.0,Provence,Bandol,NaN,Provence red blend,Domaine de la Bégude


In [39]:
wines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150930 entries, 0 to 150929
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Unnamed: 0   150930 non-null  int64  
 1   country      150925 non-null  object 
 2   description  150930 non-null  object 
 3   designation  105195 non-null  object 
 4   points       150930 non-null  int64  
 5   price        137235 non-null  float64
 6   province     150925 non-null  object 
 7   region_1     125870 non-null  object 
 8   region_2     60953 non-null   object 
 9   variety      150930 non-null  object 
 10  winery       150930 non-null  object 
dtypes: float64(1), int64(2), object(8)
memory usage: 12.7+ MB


In [41]:
for col in wines_df.select_dtypes('object').columns:
  print(f"Feature: {col}. Valores únicos: {wines_df[col].nunique()}")
  print("")

Feature: country. Valores únicos: 48

Feature: description. Valores únicos: 97821

Feature: designation. Valores únicos: 30621

Feature: province. Valores únicos: 455

Feature: region_1. Valores únicos: 1236

Feature: region_2. Valores únicos: 18

Feature: variety. Valores únicos: 632

Feature: winery. Valores únicos: 14810



### Preparación de los datos

In [67]:
wines_df_selection = wines_df[["description", "points"]].dropna().sample(2000, random_state=42)  # subset por velocidad

In [68]:
embeddings = np.vstack([get_embedding(t) for t in tqdm(wines_df_selection["description"], desc="Generating BERT embeddings")])


Generating BERT embeddings: 100%|██████████| 2000/2000 [00:24<00:00, 81.94it/s]


In [69]:
y = wines_df_selection["points"].values


In [70]:
X_train, X_test, y_train, y_test = train_test_split(embeddings, y, test_size=0.2, random_state=42)

### Entrenamiento

**Random Forest**

In [75]:
print("\n🌲 Random Forest")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


🌲 Random Forest


**XGBoost**

In [71]:
print("\n🚀 XGBoost")
xgb = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)


🚀 XGBoost


**SVM**

La *Máquina de Soporte Vectorial (SVM)* es un algoritmo supervisado que busca encontrar el **hiperplano óptimo** que mejor separa los datos en el espacio de características. 📈✨ En el caso de regresión, su versión llamada *Support Vector Regression (SVR)* intenta ajustar una línea o superficie que prediga los valores continuos dentro de un margen de tolerancia (ε).  


La SVM busca el equilibrio entre **simplicidad del modelo** (margen grande) y **precisión de predicción**, siendo especialmente útil cuando hay pocos datos o relaciones complejas entre las características. 🤖


In [72]:
print("\n📈 Support Vector Machine")
svr = SVR(kernel="rbf", C=2.0, epsilon=0.2)
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)


📈 Support Vector Machine


### Evaluación de modelos

In [76]:
# 7️⃣ Evaluación
def evaluar(nombre, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{nombre}: MSE={rmse:.2f}, R²={r2:.3f}")



In [77]:
print("\n📊 Resultados finales")
evaluar("Random Forest", y_test, y_pred_rf)
evaluar("XGBoost", y_test, y_pred_xgb)
evaluar("SVM", y_test, y_pred_svr)


📊 Resultados finales
Random Forest: MSE=5.80, R²=0.427
XGBoost: MSE=5.46, R²=0.461
SVM: MSE=4.62, R²=0.544


## Para cerrar 💬🤔

----

1. Explica en tus palabras los conceptos de:
    - Bolsa de palabras
    - Tf-IdF
    - Embbedings
2. Cómo agregarías más características para el modelo `wines prediction`


## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta el proyecto nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [Sprint 15](https://discord.com/channels/1081207584104656986/1270074296395497513).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal `#project` para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨